# Produce AAE funtional area aggregations

This notebook produces aggregations for A&E for a given NHP Demand and Capacity model scenario. Please note the following ESSENTIAL requirements for running this notebook:

- The scenario must have been run with full model results. Provide the path to the full model results in the notebook widget at the top of the notebook, in the format `full-model-results/vx.x/PROVIDER/SCENARIO_NAME/SCENARIO_RUNTIME/`
- A .env file with the correct environment variables
- Run the `generate_token.ps1` file in your local machine Terminal _before_ running this notebook, which generates SAS tokens valid for 24 hours and sets them as Databricks secrets

## Setup

In [0]:
%cd ..
%pip install .

from databricks.connect import DatabricksSession
from databricks.sdk import WorkspaceClient

from nhp.functional_areas.loading_helpers import load_env_vars, extract_model_run_details, load_op_aae_data, validate_result_path
from nhp.functional_areas.aae import process_aae, process_sdec_converted
from nhp.functional_areas.processing_helpers import qa_results
from nhp.functional_areas.saving_helpers import upload_data, add_metadata_to_ats
from datetime import datetime 
import uuid

spark = DatabricksSession.builder.getOrCreate()
w = WorkspaceClient()
dbutils = w.dbutils
dbutils.library.restartPython()

dbutils.widgets.text("capacity_model_version", "dev", "Capacity Model version")
dbutils.widgets.text("path_to_full_model_results", "", "Path to full model results")

mapping_runtime = datetime.now().strftime(format="%Y%m%d-%H%M%S")

In [0]:
path_to_full_model_results = dbutils.widgets.get('path_to_full_model_results')

db_path_to_full_model_results = str(validate_result_path(path_to_full_model_results))

env_vars = load_env_vars()

demand_model_version, fyear, provider, scenario_name, scenario_runtime = extract_model_run_details(path_to_full_model_results)

## Load data

In [0]:
aae_original = load_op_aae_data(demand_model_version, "aae", fyear, provider)
aae_model_results = spark.read.parquet(db_path_to_full_model_results + "aae")
sdec_groupings_per_run = process_sdec_converted(db_path_to_full_model_results)

## Produce aggregations

In [0]:
final_df, summary = process_aae(aae_original, aae_model_results, sdec_groupings_per_run)

## QA check

In [0]:
qa_results(aae_model_results, final_df, "arrivals")

## Upload results

In [0]:
storage_guid = str(uuid.uuid4())
metadata = {
    'PartitionKey': dbutils.widgets.get('capacity_model_version'),
    'RowKey': storage_guid,
    'app_version': demand_model_version,
    'scenario_name': scenario_name,
    'scenario_runtime': scenario_runtime,
    'dataset': provider,
    'mapping_runtime': mapping_runtime,
    'path_to_full_results_dir': path_to_full_model_results
}

upload_data(env_vars, metadata, final_df, "aae")
upload_data(env_vars, metadata, summary, "aae_summary")

## Add details to Azure Table Storage

In [0]:
add_metadata_to_ats(env_vars, metadata)